# NeurIPS 2026 Smart Buildings — Version 17

**Experiment:** online **extremum-seeking control (ESC)** of each AHU's duct static-pressure trim, built on the scored V7-style thermodynamic controller (V7 public score **0.03947**, lower is better). V17 itself is **unscored**.

## Core method

Every eligible five-minute step issues an 80 Pa positive or negative pressure probe around a slowly adapted center. Four comparison blocks follow **+ / − / − / + (ABBA)**, each 12–16 samples (depending on AHU), with the first four samples of a block excluded to allow thermal settling. Compare mean measured comfort loss plus a commanded-pressure fan-energy surrogate, demodulate by probe amplitude, low-pass the gradient, then take a projected 40 Pa maximum center update. All observations are credited to the previously issued action, not the same step.

ESC uses per-AHU constant-size state and is **not** V10's trim-and-respond on demand thresholds, V15's offline CMA-ES/GP search, an RLS learner, or MPC. The plant-physics baseline, thermal mixing, water logic, and metadata-based action layout remain independent of ESC. `TransferPolicy.ENABLE_ESC = False` disables the experiment for fixed-controller ablation.

## Safeguards and limitations

- Clip the learned center to ±250 Pa; dither only when both probe signs fit the [200, 2000] Pa physical pressure interval.
- Suspend exploration during severe (>1.8 K) comfort excursions, idle periods, invalid readings, setpoint changes, and saturation; incomplete comparison cycles are discarded. Episode `reset()` clears all learning state.
- Objective energy term is **not measured kWh**; it is only a pressure-command proxy. Building response is delayed, weather and occupancy are nonstationary, and AHUs are coupled. ABBA cancels linear time drift *only under its assumptions*; it does not prove unbiased causal gradients.
- Each four-block estimate requires roughly 4–5.3 hours of eligible AHU time. No official Kaggle scoring or full simulator validation has been completed. Test both `ENABLE_ESC` settings with matched hot/cold/mixed 30-day scenarios and compare comfort, true energy, and excursions.

## Submission

The controller is self-contained in this notebook, requires only NumPy plus the official Kaggle gateway, and returns `continuous_action` (`float32`, [-1,1]) and `discrete_action` (`int32`, {0,1}). The last cell starts `SmartBuildingsInferenceServer(TransferPolicy).run()`. Run `test_version17.py` for synthetic contract and estimator tests before simulator rollout.


In [ ]:
"""Version 17: bounded online extremum-seeking fan-pressure control.

Keep Version 7's thermodynamic feed-forward air/water dispatch and apply a
slow, cycle-synchronous extremum seeker to the fan pressure trim. No simulator
training, derivative model, PID integrator or online nonlinear search.
Only current/past observations and prior issued actions are used.

Runtime: O(number of AHUs + number of zones) per five-minute decision.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


# Fixed physics-controller coefficients (V7-style baseline). Only the fan
# pressure *trim* is learned online; these coefficients never change.
BASELINE_GAINS = {
    "active_threshold_k": 0.18,
    "cooling_offset_k": 4.0,
    "cooling_error_gain": 1.3,
    "cool_airflow_gain": 2.1,
    "reheat_gain": 0.31,
    "fan_pressure_gain_pa": 1250.0,
    "water_weather_gain": 0.35,
    "water_shortfall_gain": 5.0,
}


class PressureExtremumSeeker:
    """Four-block +,-,-,+ synchronous-dither extremum seeker.

    A/B/B/A ordering cancels linear drift in measured objective over one
    completed cycle. Ignore early samples of each block to let the slow
    building respond. The objective is observed one step *after* an action
    is issued, avoiding same-step action/reward misattribution.

    The objective uses measured comfort error and a commanded-pressure
    *energy proxy*, not true electrical kWh. This is an experiment; a
    simulator ablation must determine whether the trim helps.
    """

    SIGNS = (1, -1, -1, 1)
    DITHER_PA = 80.0
    MAX_ABS_CENTER_PA = 250.0
    MAX_UPDATE_PA = 40.0
    LEARNING_GAIN = 50000.0
    FAN_PROXY_WEIGHT = 0.08

    def __init__(self, block_steps=12, settling_steps=4):
        if block_steps <= settling_steps or settling_steps < 0:
            raise ValueError("Each perturbation block needs scored steps")
        self.block_steps = int(block_steps)
        self.settling_steps = int(settling_steps)
        self.reset()

    def reset(self):
        """Forget all adaptation and pending measurements between episodes."""
        self.center_pa = 0.0
        self.gradient_ema = 0.0
        self.cycles_completed = 0
        self._abort_cycle()

    def _abort_cycle(self):
        """Discard incomplete comparisons without discarding a learned trim."""
        self.block = 0
        self.steps = 0
        self.samples = []
        self.block_costs = []
        self.pending_pressure_pa = None
        self.pending_context = None

    def _complete_block(self):
        self.block_costs.append(sum(self.samples) / len(self.samples))
        self.samples = []
        self.steps = 0
        self.block += 1
        if self.block != len(self.SIGNS):
            return

        plus = 0.5 * (self.block_costs[0] + self.block_costs[3])
        minus = 0.5 * (self.block_costs[1] + self.block_costs[2])
        gradient = (plus - minus) / (2.0 * self.DITHER_PA)
        self.gradient_ema = 0.5 * self.gradient_ema + 0.5 * gradient
        update = float(np.clip(
            self.LEARNING_GAIN * self.gradient_ema,
            -self.MAX_UPDATE_PA, self.MAX_UPDATE_PA))
        self.center_pa = float(np.clip(
            self.center_pa - update,
            -self.MAX_ABS_CENTER_PA, self.MAX_ABS_CENTER_PA))
        self.cycles_completed += 1
        self.block = 0
        self.block_costs = []

    def choose_pressure(self, baseline_pa, comfort_loss, context, eligible):
        """Ingest prior action's measured result; issue this step's pressure.

        'context' holds zone setpoint bands. A band change or unsafe
        operating condition breaks the comparison rather than mixing
        occupied/unoccupied energy and comfort objectives.
        """
        baseline_pa = float(baseline_pa)
        if not eligible or not np.isfinite(comfort_loss):
            self._abort_cycle()
            return baseline_pa

        if self.pending_pressure_pa is not None:
            if context != self.pending_context:
                self._abort_cycle()
            else:
                # The current observation reflects the *previous* command.
                cost = float(comfort_loss) + self.FAN_PROXY_WEIGHT * (
                    self.pending_pressure_pa / 1000.0) ** 1.5
                self.steps += 1
                if self.steps > self.settling_steps:
                    self.samples.append(cost)
                if self.steps == self.block_steps:
                    self._complete_block()

        # Projection cannot recover an unbiased gradient at actuator limits.
        # Suspend the whole cycle rather than comparing clipped probes.
        low_probe = baseline_pa + self.center_pa - self.DITHER_PA
        high_probe = baseline_pa + self.center_pa + self.DITHER_PA
        if low_probe < 200.0 or high_probe > 2000.0:
            self._abort_cycle()
            return baseline_pa

        proposal = baseline_pa + self.center_pa + self.SIGNS[self.block] * self.DITHER_PA
        self.pending_pressure_pa = proposal
        self.pending_context = context
        return proposal


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15
    ENABLE_ESC = True  # Set False for the exact fixed-gain ablation.

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.p = BASELINE_GAINS
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outdoor_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outdoor_mean = float(outdoor_stats.get("sample_mean", float("nan")))
        self.outdoor_std = float(outdoor_stats.get("sample_std", float("nan")))

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Reset per-AHU dither phases, feedback buffers and learned trims."""
        self.seekers = [
            PressureExtremumSeeker(block_steps=12 + 2 * (i % 3))
            for i in range(self.ahu_count)
        ]

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Denormalize outdoor air; never treat a z-score as Kelvin."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outdoor_mean) and
                np.isfinite(self.outdoor_std) and self.outdoor_std > 0.0):
            kelvin = raw * self.outdoor_std + self.outdoor_mean
        else:
            kelvin = raw
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
            "low": low,
            "high": high,
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.p["active_threshold_k"]:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - self.p["cooling_offset_k"] - self.p["cooling_error_gain"] * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.p["active_threshold_k"]
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.p["active_threshold_k"]:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + self.p["cool_airflow_gain"] * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.p["active_threshold_k"]:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + self.p["reheat_gain"] * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Apply safe V7 physics plus a measured, slow fan-pressure dither."""
        seeker = self.seekers[ahu["run"]]
        if not statuses:
            seeker._abort_cycle()
            return

        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.p["active_threshold_k"]:
            seeker._abort_cycle()
            return

        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)

        if ahu["pressure"] is None:
            seeker._abort_cycle()
            return

        demand_proxy = max(mean_damper, 0.50 * peak_damper)
        baseline_pa = self.MIN_FAN_PRESSURE_PA + self.p["fan_pressure_gain_pa"] * np.sqrt(
            self._clamp(
                (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                0.0, 1.0))

        # Safety gate: no excitation while zones are badly out of range,
        # after missing measurements, or when the AHU is not conditioning.
        # All setpoint windows must remain the same across a probe cycle.
        eligible = (
            self.ENABLE_ESC
            and len(statuses) == len(ahu["zones"])
            and self.p["active_threshold_k"] <= worst_violation_k <= 1.8
            and all(np.isfinite(z["temp"]) for z in statuses)
        )
        comfort_loss = sum(
            z["hot"] ** 2 + z["cold"] ** 2 for z in statuses
        ) / len(statuses)
        context = tuple((round(z["low"], 2), round(z["high"], 2))
                        for z in statuses)
        pressure_pa = seeker.choose_pressure(
            baseline_pa, comfort_loss, context, eligible)
        continuous[ahu["pressure"]] = self._clamp(
            pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + self.p["water_weather_gain"] * max(0.0, 294.0 - outside) + self.p["water_shortfall_gain"] * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """One-pass physics dispatch; online ESC consumes delayed sensor feedback.

        Perturbations are only applied when the comparison is safe and
        informative. Otherwise the original fixed physics action is used.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, z, fallback) for z in ahu["zones"]
            ]
            statuses = [s for s in statuses if s is not None]
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
